Load and Explore the Data

In [79]:
import numpy as np
import pandas as pd
import ast
import pickle
import warnings
warnings.filterwarnings('ignore')

from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully ✓")

All libraries imported successfully ✓


In [80]:
# Load datasets
movie   = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv', usecols=[0, 1, 2, 3], header=0)
credits.columns = ['movie_id', 'title', 'cast', 'crew']

print("Movies dataset shape  :", movie.shape)
print("Credits dataset shape :", credits.shape)
print("\nMovies columns:", movie.columns.tolist())

Movies dataset shape  : (4803, 20)
Credits dataset shape : (3863, 4)

Movies columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


In [81]:
# Quick exploration
print("=== Movies Dataset ===")
display(movie.head(3))

print("\n=== Credits Dataset ===")
display(credits[['movie_id', 'title', 'cast', 'crew']].head(3))

print("\n=== Missing Values (Movies) ===")
print(movie.isnull().sum()[movie.isnull().sum() > 0])

=== Movies Dataset ===


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466



=== Credits Dataset ===


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."



=== Missing Values (Movies) ===
homepage        3091
overview           3
release_date       1
runtime            2
tagline          844
dtype: int64


In [82]:
# Fix type mismatch and merge
credits['movie_id'] = pd.to_numeric(credits['movie_id'], errors='coerce')
credits.dropna(subset=['movie_id'], inplace=True)
credits['movie_id'] = credits['movie_id'].astype(int)

# Merge both datasets on movie id
movies = movie.merge(credits, left_on='id', right_on='movie_id')

# Keep important columns
movies = movies[['id', 'title_x', 'overview', 'keywords', 'genres', 'cast', 'crew']].copy()
movies.rename(columns={'title_x': 'title'}, inplace=True)

# Drop rows with null values
movies.dropna(inplace=True)

print("Merged dataset shape:", movies.shape)
display(movies.head(3))

Merged dataset shape: (1487, 7)


,id,title,overview,keywords,genres,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


Content-Based Filtering - Feature Extraction

In [83]:
# Process 'overview' column - split into word tokens
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])
print("Sample overview tokens:", movies['overview'].iloc[0][:8])

Sample overview tokens: ['In', 'the', '22nd', 'century,', 'a', 'paraplegic', 'Marine', 'is']


In [84]:
# Process 'keywords' column - extract name field from JSON list
def convert_keywords(obj):
    try:
        return [i['name'] for i in ast.literal_eval(obj)]
    except:
        return []

movies['keywords'] = movies['keywords'].apply(convert_keywords)
print("Sample keywords:", movies['keywords'].iloc[0])

Sample keywords: ['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien planet', 'cgi', 'marine', 'soldier', 'battle', 'love affair', 'anti war', 'power relations', 'mind and soul', '3d']


In [85]:
# Process 'genres' column
def convert_genres(obj):
    try:
        return [i['name'] for i in ast.literal_eval(obj)]
    except:
        return []

movies['genres'] = movies['genres'].apply(convert_genres)
print("Sample genres:", movies['genres'].iloc[0])

Sample genres: ['Action', 'Adventure', 'Fantasy', 'Science Fiction']


In [86]:
# Process 'cast' column - top 3 actors only
def extract_cast(obj):
    try:
        return [i['name'] for i in ast.literal_eval(obj)[:3]]
    except:
        return []

movies['cast'] = movies['cast'].apply(extract_cast)
print("Sample cast (top 3):", movies['cast'].iloc[0])

Sample cast (top 3): ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']


In [87]:
# Process 'crew' column - extract Director only
def extract_director(obj):
    try:
        for i in ast.literal_eval(obj):
            if i['job'] == 'Director':
                return [i['name']]
    except:
        pass
    return []

movies['crew'] = movies['crew'].apply(extract_director)
print("Sample director:", movies['crew'].iloc[0])

Sample director: ['James Cameron']


In [88]:
# Combine all features into a single 'tags' column
movies['tags'] = (
    movies['overview'] +
    movies['keywords'] +
    movies['genres']   +
    movies['cast']     +
    movies['crew']
)

# Remove spaces within multi-word tokens so they become single tokens
movies['tags'] = movies['tags'].apply(lambda x: [i.replace(" ", "") for i in x])

# Keep only needed columns
movies = movies[['id', 'title', 'tags']].reset_index(drop=True)
print("Sample tags (first 10 tokens):", movies['tags'].iloc[0][:100])

Sample tags (first 10 tokens): ['In', 'the', '22nd', 'century,', 'a', 'paraplegic', 'Marine', 'is', 'dispatched', 'to', 'the', 'moon', 'Pandora', 'on', 'a', 'unique', 'mission,', 'but', 'becomes', 'torn', 'between', 'following', 'orders', 'and', 'protecting', 'an', 'alien', 'civilization.', 'cultureclash', 'future', 'spacewar', 'spacecolony', 'society', 'spacetravel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alienplanet', 'cgi', 'marine', 'soldier', 'battle', 'loveaffair', 'antiwar', 'powerrelations', 'mindandsoul', '3d', 'Action', 'Adventure', 'Fantasy', 'ScienceFiction', 'SamWorthington', 'ZoeSaldana', 'SigourneyWeaver', 'JamesCameron']


In [89]:
# Stemming - reduce words to their root form
ps = PorterStemmer()

def stemming(text_list):
    return " ".join(ps.stem(word) for word in text_list)

movies['tags'] = movies['tags'].apply(stemming)
movies['tags'] = movies['tags'].apply(lambda x: x.lower())

print("Sample tag string:")
print(movies['tags'].iloc[0][:200], "...")

Sample tag string:
in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. cultureclash futur spacewar spacecoloni ...


In [90]:
# Vectorization using CountVectorizer (Bag of Words)
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectors    = vectorizer.fit_transform(movies['tags']).toarray()

print("Vocabulary size :", len(vectorizer.vocabulary_))
print("Vector matrix   :", vectors.shape)
print("\nSample vocabulary (first 20 words):")
print(sorted(vectorizer.vocabulary_.keys())[:20])

Vocabulary size : 5000
Vector matrix   : (1487, 5000)

Sample vocabulary (first 20 words):
['000', '007', '10', '11', '12', '12th', '13', '14', '14th', '15', '150', '15thcenturi', '16', '16th', '17', '1863', '18th', '18thcenturi', '19', '1910']


In [91]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [92]:
# Compute Cosine Similarity matrix
similarity = cosine_similarity(vectors)

print("Similarity matrix shape:", similarity.shape)
print("\nSample similarity scores for movie[0] (Avatar) - top 5:")
scores = sorted(enumerate(similarity[0]), key=lambda x: x[1], reverse=True)[1:6]
for idx, score in scores:
    print(f"  {movies.iloc[idx].title:<45} score: {score:.4f}")

Similarity matrix shape: (1487, 1487)

Sample similarity scores for movie[0] (Avatar) - top 5:
  Aliens vs Predator: Requiem                   score: 0.2590
  Titan A.E.                                    score: 0.2354
  Independence Day                              score: 0.2343
  Battle: Los Angeles                           score: 0.2262
  Predators                                     score: 0.2248


In [93]:
# sorted(enumerate(similarity[0]))
similarity

array([[1.        , 0.07624929, 0.08087458, ..., 0.        , 0.02990743,
        0.        ],
       [0.07624929, 1.        , 0.05892557, ..., 0.03149704, 0.03268602,
        0.        ],
       [0.08087458, 0.05892557, 1.        , ..., 0.        , 0.06933752,
        0.        ],
       ...,
       [0.        , 0.03149704, 0.        , ..., 1.        , 0.        ,
        0.086711  ],
       [0.02990743, 0.03268602, 0.06933752, ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.086711  , 0.        ,
        1.        ]])

In [94]:
movies

,id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."
...,...,...,...
1482,10685,The Watcher,"fbi agent joel campbell, burnt-out and shell-s..."
1483,7220,The Punisher,when undercov fbi agent frank castle' wife and...
1484,9763,Goal!: The Dream Begins,"like million of kid around the world, santiago..."
1485,72387,Safe,after a former elit agent rescu a 12-year-old ...


Save the Model

In [95]:
# Save the processed movie dataframe and similarity matrix
pickle.dump(movies,     open('model.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("model.pkl      saved ✓  (shape:", movies.shape, ")")
print("similarity.pkl saved ✓  (shape:", similarity.shape, ")")

model.pkl      saved ✓  (shape: (1487, 3) )
similarity.pkl saved ✓  (shape: (1487, 1487) )


Test Recommendations

In [96]:
def Recommendation_system(movie_title, n=10):
    """
    Returns top-n content-based recommendations for a given movie title.
    """
    matches = movies[movies['title'] == movie_title]
    if matches.empty:
        print(f"Movie '{movie_title}' not found in the dataset.")
        return []

    movie_index = matches.index[0]
    distances   = sorted(
        list(enumerate(similarity[movie_index])),
        reverse=True,
        key=lambda x: x[1]
    )

    recommendations = []
    for i in distances[1:n+1]:
        recommendations.append(movies.iloc[i[0]].title)

    return recommendations

In [97]:
# Test the recommendation function
test_movies = ['Avatar', 'Interstellar', 'The Dark Knight', 'The Avengers', 'Inception']

for title in test_movies:
    recs = Recommendation_system(title)
    print(f"\n Because you watched: '{title}'")
    print("   You might also like:")
    for j, r in enumerate(recs, 1):
        print(f"   {j:2}. {r}")


 Because you watched: 'Avatar'
   You might also like:
    1. Aliens vs Predator: Requiem
    2. Titan A.E.
    3. Independence Day
    4. Battle: Los Angeles
    5. Predators
    6. Jupiter Ascending
    7. Small Soldiers
    8. Meet Dave
    9. Edge of Tomorrow
   10. The Fifth Element

 Because you watched: 'Interstellar'
   You might also like:
    1. Guardians of the Galaxy
    2. Apollo 13
    3. The Martian
    4. Space Cowboys
    5. A.I. Artificial Intelligence
    6. Ender's Game
    7. Zathura: A Space Adventure
    8. Contact
    9. Solaris
   10. Escape from Planet Earth

 Because you watched: 'The Dark Knight'
   You might also like:
    1. The Dark Knight Rises
    2. Batman Begins
    3. Batman Returns
    4. Batman Forever
    5. Batman
    6. Batman & Robin
    7. Batman v Superman: Dawn of Justice
    8. Law Abiding Citizen
    9. Punisher: War Zone
   10. Sherlock Holmes: A Game of Shadows

 Because you watched: 'The Avengers'
   You might also like:
    1. Avenger

---
User-Based Collaborative Filtering

We use the simulated user ratings data to find users with similar taste and recommend movies they liked.

In [98]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# --- Simulated user-movie ratings matrix (rows=users, cols=movies) ---
# Rating scale: 1-5, 0 means not rated
np.random.seed(42)
num_users  = 10
num_movies = 15

movie_titles_sample = [
    'Avatar', 'Interstellar', 'The Dark Knight', 'Inception',
    'The Avengers', 'Iron Man', 'Toy Story', 'Forrest Gump',
    'The Matrix', 'Pulp Fiction', 'Titanic', 'Jurassic Park',
    'Gladiator', 'The Lion King', 'Schindler\'s List'
]

# Generate sparse ratings (many 0s = not watched)
ratings_matrix = np.random.choice([0, 0, 0, 1, 2, 3, 4, 5],
                                   size=(num_users, num_movies))

ratings_df = pd.DataFrame(
    ratings_matrix,
    index   = [f'User_{i+1}' for i in range(num_users)],
    columns = movie_titles_sample
)

print("User-Movie Ratings Matrix:")
display(ratings_df)

User-Movie Ratings Matrix:


,Avatar,Interstellar,The Dark Knight,Inception,The Avengers,Iron Man,Toy Story,Forrest Gump,The Matrix,Pulp Fiction,Titanic,Jurassic Park,Gladiator,The Lion King,Schindler's List
User_1,4,1,2,4,0,5,2,2,4,0,0,4,0,0,5
User_2,2,1,5,5,0,3,2,0,5,1,3,3,0,5,1
User_3,2,0,1,0,3,2,1,0,0,0,0,4,0,5,1
User_4,1,5,4,3,3,4,3,0,1,4,1,5,0,0,2
User_5,0,4,2,0,4,0,1,0,1,3,0,0,0,0,2
User_6,0,1,1,4,1,4,1,2,5,4,0,3,0,1,0
User_7,5,1,0,3,3,3,0,1,3,2,4,0,0,1,0
User_8,0,3,1,3,4,5,4,5,3,4,1,0,3,5,2
User_9,5,2,0,4,2,5,0,0,1,1,1,2,0,2,4
User_10,2,0,0,4,0,5,0,1,5,5,4,0,0,0,5


In [99]:
def user_based_recommend(target_user, ratings_df, top_n_users=3, top_n_movies=5):
    """
    User-Based Collaborative Filtering.
    Finds users similar to target_user using cosine similarity,
    then recommends movies those similar users rated highly
    but the target user hasn't seen yet.
    """
    # Compute cosine similarity between all users
    user_sim_matrix = cos_sim(ratings_df.values)
    user_sim_df     = pd.DataFrame(
        user_sim_matrix,
        index   = ratings_df.index,
        columns = ratings_df.index
    )

    # Find top-N similar users (excluding the target user itself)
    similar_users = (
        user_sim_df[target_user]
        .drop(target_user)
        .sort_values(ascending=False)
        .head(top_n_users)
        .index.tolist()
    )

    # Movies the target user has already watched (rated > 0)
    already_watched = ratings_df.loc[target_user][ratings_df.loc[target_user] > 0].index.tolist()

    # Aggregate scores from similar users for unseen movies
    score_accumulator = {}
    for sim_user in similar_users:
        sim_score = user_sim_df.loc[target_user, sim_user]
        for movie_col in ratings_df.columns:
            if movie_col not in already_watched and ratings_df.loc[sim_user, movie_col] > 0:
                if movie_col not in score_accumulator:
                    score_accumulator[movie_col] = 0
                score_accumulator[movie_col] += sim_score * ratings_df.loc[sim_user, movie_col]

    # Sort by weighted score and return top-N
    recommendations = sorted(score_accumulator.items(), key=lambda x: x[1], reverse=True)
    return [m for m, _ in recommendations[:top_n_movies]]


# Test collaborative filtering for each user
for user in ratings_df.index[:5]:
    recs = user_based_recommend(user, ratings_df)
    watched = ratings_df.loc[user][ratings_df.loc[user] > 0].index.tolist()
    print(f"\n {user}")
    print(f"   Already watched : {watched}")
    print(f"   Recommendations : {recs}")


 User_1
   Already watched : ['Avatar', 'Interstellar', 'The Dark Knight', 'Inception', 'Iron Man', 'Toy Story', 'Forrest Gump', 'The Matrix', 'Jurassic Park', "Schindler's List"]
   Recommendations : ['Titanic', 'The Lion King', 'Pulp Fiction', 'The Avengers']

 User_2
   Already watched : ['Avatar', 'Interstellar', 'The Dark Knight', 'Inception', 'Iron Man', 'Toy Story', 'The Matrix', 'Pulp Fiction', 'Titanic', 'Jurassic Park', 'The Lion King', "Schindler's List"]
   Recommendations : ['Forrest Gump', 'The Avengers']

 User_3
   Already watched : ['Avatar', 'The Dark Knight', 'The Avengers', 'Iron Man', 'Toy Story', 'Jurassic Park', 'The Lion King', "Schindler's List"]
   Recommendations : ['Inception', 'The Matrix', 'Interstellar', 'Pulp Fiction', 'Titanic']

 User_4
   Already watched : ['Avatar', 'Interstellar', 'The Dark Knight', 'Inception', 'The Avengers', 'Iron Man', 'Toy Story', 'The Matrix', 'Pulp Fiction', 'Titanic', 'Jurassic Park', "Schindler's List"]
   Recommendations 

---
Hybrid Recommender

Combines both **Content-Based** scores with **User-Based Collaborative Filtering** scores using a weighted average.

In [100]:
def hybrid_recommend(movie_title, target_user, ratings_df,
                     content_weight=0.6, collab_weight=0.4, top_n=10):
    """
    Hybrid Recommender: blends content-based and collaborative filtering.

    Parameters
    ----------
    movie_title    : str   – seed movie for content-based filtering
    target_user    : str   – user ID for collaborative filtering
    content_weight : float – weight for content-based scores  (default 0.6)
    collab_weight  : float – weight for collaborative scores  (default 0.4)
    top_n          : int   – number of recommendations to return
    """
    assert content_weight + collab_weight == 1.0, "Weights must sum to 1.0"

    # ── Content-Based scores ─────────────────────────────────────────────
    content_scores = {}
    matches = movies[movies['title'] == movie_title]
    if not matches.empty:
        idx = matches.index[0]
        for i, score in enumerate(similarity[idx]):
            if i != idx:
                content_scores[movies.iloc[i].title] = score

    # ── Collaborative-Filtering scores ───────────────────────────────────
    collab_recs  = user_based_recommend(target_user, ratings_df, top_n_movies=len(ratings_df.columns))
    collab_scores = {}
    for rank, title in enumerate(collab_recs):
        # Higher-ranked = higher score; simple reciprocal rank scoring
        collab_scores[title] = 1 / (rank + 1)

    # Normalise to [0, 1]
    def normalise(d):
        if not d: return d
        max_v = max(d.values())
        return {k: v / max_v for k, v in d.items()} if max_v > 0 else d

    content_norm = normalise(content_scores)
    collab_norm  = normalise(collab_scores)

    # ── Combine ──────────────────────────────────────────────────────────
    all_titles = set(content_norm) | set(collab_norm)
    hybrid_scores = {}
    for t in all_titles:
        c = content_norm.get(t, 0)
        u = collab_norm.get(t, 0)
        hybrid_scores[t] = content_weight * c + collab_weight * u

    # Sort and exclude the seed movie itself
    sorted_recs = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    return [(t, round(s, 4)) for t, s in sorted_recs if t != movie_title][:top_n]


# Demo the hybrid recommender
seed_movie  = 'Interstellar'
sample_user = 'User_1'

print(f" Hybrid Recommendations")
print(f"   Seed movie  : '{seed_movie}'")
print(f"   Target user : {sample_user}")
print(f"   Weights     : Content=0.6 | Collaborative=0.4")
print()

hybrid_recs = hybrid_recommend(seed_movie, sample_user, ratings_df)
for rank, (title, score) in enumerate(hybrid_recs, 1):
    print(f"  {rank:2}. {title:<45} hybrid score: {score}")

 Hybrid Recommendations
   Seed movie  : 'Interstellar'
   Target user : User_1
   Weights     : Content=0.6 | Collaborative=0.4

   1. Guardians of the Galaxy                       hybrid score: 0.6
   2. Apollo 13                                     hybrid score: 0.5425
   3. The Martian                                   hybrid score: 0.5374
   4. Space Cowboys                                 hybrid score: 0.5237
   5. A.I. Artificial Intelligence                  hybrid score: 0.5224
   6. Ender's Game                                  hybrid score: 0.5077
   7. Zathura: A Space Adventure                    hybrid score: 0.5056
   8. Contact                                       hybrid score: 0.4986
   9. Solaris                                       hybrid score: 0.4787
  10. Escape from Planet Earth                      hybrid score: 0.4749


---
## Summary

| Approach | Method | Strengths |
|---|---|---|
| **Content-Based** | CountVectorizer + Cosine Similarity | Works with no user history |
| **Collaborative** | User similarity via Cosine Similarity | Captures user taste patterns |
| **Hybrid** | Weighted blend of both | Most robust & accurate |

**Model files saved:**
- `model.pkl` - processed movies DataFrame
- `similarity.pkl` - cosine similarity matrix

These are loaded by `app.py` to serve recommendations via Flask.